In [1]:
import torch
import snntorch as snn
from matplotlib import pyplot as plt
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from torch import nn
import numpy as np

import sys
sys.path.append(f"../model_compiler")
from neuron_mapper import Neuron_Mapper, Neuron_LIF

In [2]:
# --- Configurations ---
CUT_TRAINING_SHORT = False
TRAINING_CUTOFF = 0.1

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float32

# MNIST specific parameters
input_nodes = 28 * 28      # 784 pixels
hidden_neurons = 128       # Hidden layer size
output_neurons = 10        # 10 digit classes (0-9)

threshold = 1.0            # LIF neuron threshold
beta = Neuron_LIF.DECAY_MODE_LIF_2  # LIF neuron decay rate
reset_mode = Neuron_LIF.RESET_MODE_ZERO  # Reset mode for LIF neuron
dt_steps = 50              # Time steps (rate coding duration)
epochs = 10                # Training epochs
batch_size = 64            # Batch size for training
learning_rate = 5e-4       # Learning rate

In [3]:
# --- Load and Prepare MNIST Data ---
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST normalization
])

# Load MNIST dataset
train_dataset = torchvision.datasets.MNIST(
    root='./data', 
    train=True, 
    download=True, 
    transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root='./data', 
    train=False, 
    download=True, 
    transform=transform
)

In [4]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [5]:
def rate_encode_poisson(inputs, num_steps):
    """
    inputs: [batch, features] — normalized input
    num_steps: number of time steps
    returns: [num_steps, batch, features] — binary spikes
    """
    # Normalize input to [0, 1] range
    inputs_norm = torch.clamp(inputs, 0, 1)
    
    # Generate spikes with probability = input value
    spike_prob = inputs_norm.unsqueeze(0).expand(num_steps, -1, -1)
    spikes = torch.rand_like(spike_prob) < spike_prob
    return spikes.float()

In [6]:
# --- Define Spiking MNIST Model ---
class SpikingMNISTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.input = nn.Linear(input_nodes, hidden_neurons, bias=False)
        self.lif1 = snn.Leaky(beta=beta, threshold=threshold)
        self.transfer1 = nn.Linear(hidden_neurons, output_neurons, bias=False)
        self.output = snn.Leaky(beta=beta, threshold=threshold)

    def forward(self, x):
        # Reshape input from [batch, 1, 28, 28] to [batch, 784]
        x = x.view(x.size(0), -1)
        
        # Rate encode the input
        x_spike_seq = rate_encode_poisson(x, dt_steps).to(device)
        
        mem1 = self.lif1.init_leaky()
        mem2 = self.output.init_leaky()

        spike_record = []

        for t in range(dt_steps):
            x_t = x_spike_seq[t]
            cur1 = self.input(x_t)
            spk1, mem1 = self.lif1(cur1, mem1)

            cur2 = self.transfer1(spk1)
            spk2, mem2 = self.output(cur2, mem2)

            spike_record.append(spk2)

        # Accumulate spikes across time
        spike_sum = torch.stack(spike_record, dim=0).sum(dim=0)  # [batch, output_neurons]
        return spike_sum

# --- Training Function ---
def train_classifier(classifier, loss_function, optimizer, train_loader):
    print("Training started...\n")
    loss_history = []

    for epoch in range(epochs):
        classifier.train()
        total_loss = 0
        num_batches = 0

        for batch_idx, (data, targets) in enumerate(train_loader):
            data, targets = data.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs = classifier(data)
            loss = loss_function(outputs, targets)
            
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            num_batches += 1

            if batch_idx % 100 == 0:
                print(f'Epoch {epoch+1}/{epochs}, Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}')

        avg_loss = total_loss / num_batches
        loss_history.append(avg_loss)
        print(f"Epoch {epoch + 1}/{epochs} - Average Loss: {avg_loss:.4f}")

        if CUT_TRAINING_SHORT and avg_loss < TRAINING_CUTOFF:
            break

    return loss_history

def test_classifier(classifier, test_loader):
    print("\nTesting started...\n")
    classifier.eval()
    
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, targets in test_loader:
            data, targets = data.to(device), targets.to(device)
            outputs = classifier(data)
            _, predicted = torch.max(outputs, 1)
            
            total += targets.size(0)
            correct += (predicted == targets).sum().item()

    return correct, total

In [7]:
# --- Run Everything ---
print(f"Using device: {device}")
classifier = SpikingMNISTClassifier().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(classifier.parameters(), lr=learning_rate)

# Train the model
train_loss = train_classifier(classifier, loss_function, optimizer, train_loader)

# Save the trained model
torch.save(classifier.state_dict(), "spiking_mnist_classifier_t512.pth")
print("Model saved to spiking_mnist_classifier.pth")

# Set model to evaluation mode if testing or using it for inference
classifier.eval()

# Now you can use it for testing, inference, etc.
correct, total = test_classifier(classifier, test_loader)
print(f"Restored Model Test Accuracy: {100 * correct / total:.2f}%")

Using device: cpu
Training started...

Epoch 1/10, Batch 0/938, Loss: 2.3026


Epoch 1/10, Batch 100/938, Loss: 0.9318


Epoch 1/10, Batch 200/938, Loss: 0.1493


Epoch 1/10, Batch 300/938, Loss: 0.4677


Epoch 1/10, Batch 400/938, Loss: 0.2328


...

Epoch 10/10, Batch 700/938, Loss: 0.0215


Epoch 10/10, Batch 800/938, Loss: 0.0887


Epoch 10/10, Batch 900/938, Loss: 0.0150


Epoch 10/10 - Average Loss: 0.0305
Model saved to spiking_mnist_classifier.pth

Testing started...



Restored Model Test Accuracy: 96.83%


In [8]:
# Recreate the model architecture
classifier = SpikingMNISTClassifier()

# Load the state dict from file
classifier.load_state_dict(torch.load("spiking_mnist_classifier_t512.pth", map_location=torch.device('cpu')))

# Move model to the same device you used before (e.g., GPU if available)
classifier.to(device)

# Set model to evaluation mode if testing or using it for inference
classifier.eval()

# Now you can use it for testing, inference, etc.
correct, total = test_classifier(classifier, test_loader)
print(f"Restored Model Test Accuracy: {100 * correct / total:.2f}%")


Testing started...



/tmp/ipykernel_41399/2854324426.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  classifier.load_state_dict(torch.load("spiking_mnist_classifier_t512.pth", map_location=t

Restored Model Test Accuracy: 96.79%


In [9]:
# --- Neuron Mapping ---
# mapping model
neuron_layers = {
    "Input" : {
        "decay_mode": beta,
        "threshold": threshold,
        "neurons": input_nodes,
        "reset_mode": reset_mode
    },
    "Layer1": {
        "decay_mode": beta,
        "threshold": threshold,
        "neurons": hidden_neurons,
        "reset_mode": reset_mode
    },
    "Layer2": {
        "decay_mode": beta,
        "threshold": threshold,
        "neurons": output_neurons,
        "reset_mode": reset_mode
    }
}

neuron_weights = {
    "Input-Layer1": {
        "weights": classifier.input.weight.detach().cpu().numpy().tolist()
    },
    "Layer1-Layer2": {
        "weights": classifier.transfer1.weight.detach().cpu().numpy().tolist()
    }
}

print("Neuron weights extracted")

Neuron weights extracted


In [10]:
neuron_mapper = Neuron_Mapper(neuron_layers, neuron_weights)
neuron_mapper.map()

# Write mapping to file
output_file = "neuron_mapping.txt"
with open(output_file, 'w') as f:
    f.write(neuron_mapper.generate_init())

print(f"Neuron mapping saved to {output_file}")

Neuron mapping saved to neuron_mapping.txt


In [11]:
neuron_mapper.log_mapping()

Virtual Cluster ID: 32
Neuron ID: 0, Cluster ID: 32, Decay Mode: 0.5, Threshold: 1.0
Neuron ID: 1, Cluster ID: 32, Decay Mode: 0.5, Threshold: 1.0
Neuron ID: 2, Cluster ID: 32, Decay Mode: 0.5, Threshold: 1.0
Neuron ID: 3, Cluster ID: 32, Decay Mode: 0.5, Threshold: 1.0
Neuron ID: 4, Cluster ID: 32, Decay Mode: 0.5, Threshold: 1.0
Neuron ID: 5, Cluster ID: 32, Decay Mode: 0.5, Threshold: 1.0
Neuron ID: 6, Cluster ID: 32, Decay Mode: 0.5, Threshold: 1.0
Neuron ID: 7, Cluster ID: 32, Decay Mode: 0.5, Threshold: 1.0
Neuron ID: 8, Cluster ID: 32, Decay Mode: 0.5, Threshold: 1.0
Neuron ID: 9, Cluster ID: 32, Decay Mode: 0.5, Threshold: 1.0
Neuron ID: 10, Cluster ID: 32, Decay Mode: 0.5, Threshold: 1.0
Neuron ID: 11, Cluster ID: 32, Decay Mode: 0.5, Threshold: 1.0
Neuron ID: 12, Cluster ID: 32, Decay Mode: 0.5, Threshold: 1.0
Neuron ID: 13, Cluster ID: 32, Decay Mode: 0.5, Threshold: 1.0
Neuron ID: 14, Cluster ID: 32, Decay Mode: 0.5, Threshold: 1.0
Neuron ID: 15, Cluster ID: 32, Decay Mode:

In [12]:
# --- Generate Test Values ---
# Take a small subset of test data for spike encoding example
test_subset = []
test_labels = []
dt_steps = 50
for i, (data, label) in enumerate(test_loader):
    if i >= 10:  # Only take first 5 batches
        break
    test_subset.append(data)
    test_labels.append(label)

test_data = torch.cat(test_subset, dim=0)  # Take first 30 samples
test_targets = torch.cat(test_labels, dim=0)

print(f"Test data shape: {test_data.shape}")

# Encode test data to spikes
test_data_flat = test_data.view(test_data.size(0), -1)
test_encoded = rate_encode_poisson(test_data_flat, dt_steps)

print(f"Test encoded shape: {test_encoded.shape}")

Test data shape: torch.Size([640, 1, 28, 28])


Test encoded shape: torch.Size([50, 640, 784])


In [13]:
# Write test values to file
with open("test_values.txt", 'w') as f:
    for sample in range(test_encoded.shape[1]):  # input samples
        print(f"Processing sample {sample + 1}/{test_encoded.shape[1]}")
        for t in range(test_encoded.shape[0]):  # time steps
            for pixel in range(test_encoded.shape[2]):  # 784 pixels
                value = test_encoded[t][sample][pixel]
                if value > 0:
                    cluster_id = (pixel // 32 ) + 32  # 5 bits (pixel ID modulo 32)
                    neuron_id = pixel % 32  # 5 bits (pixel ID modulo 32)
                    packet = (cluster_id << 5) | neuron_id  # 6-bit cluster + 5-bit neuron
                    f.write(f"{packet:03X}\n")
                else:
                    f.write("FFF\n")

print("Test values saved to mnist_test_values.txt")
print("Test labels:", test_targets.numpy())

Processing sample 1/640


Processing sample 2/640


Processing sample 3/640


Processing sample 4/640


Processing sample 5/640


...

Processing sample 637/640


Processing sample 638/640


Processing sample 639/640


Processing sample 640/640


Test values saved to mnist_test_values.txt
Test labels: [7 2 1 0 4 1 4 9 5 9 0 6 9 0 1 5 9 7 3 4 9 6 6 5 4 0 7 4 0 1 3 1 3 4 7 2 7
 1 2 1 1 7 4 2 3 5 1 2 4 4 6 3 5 5 6 0 4 1 9 5 7 8 9 3 7 4 6 4 3 0 7 0 2 9
 1 7 3 2 9 7 7 6 2 7 8 4 7 3 6 1 3 6 9 3 1 4 1 7 6 9 6 0 5 4 9 9 2 1 9 4 8
 7 3 9 7 4 4 4 9 2 5 4 7 6 7 9 0 5 8 5 6 6 5 7 8 1 0 1 6 4 6 7 3 1 7 1 8 2
 0 2 9 9 5 5 1 5 6 0 3 4 4 6 5 4 6 5 4 5 1 4 4 7 2 3 2 7 1 8 1 8 1 8 5 0 8
 9 2 5 0 1 1 1 0 9 0 3 1 6 4 2 3 6 1 1 1 3 9 5 2 9 4 5 9 3 9 0 3 6 5 5 7 2
 2 7 1 2 8 4 1 7 3 3 8 8 7 9 2 2 4 1 5 9 8 7 2 3 0 4 4 2 4 1 9 5 7 7 2 8 2
 6 8 5 7 7 9 1 8 1 8 0 3 0 1 9 9 4 1 8 2 1 2 9 7 5 9 2 6 4 1 5 8 2 9 2 0 4
 0 0 2 8 4 7 1 2 4 0 2 7 4 3 3 0 0 3 1 9 6 5 2 5 9 2 9 3 0 4 2 0 7 1 1 2 1
 5 3 3 9 7 8 6 5 6 1 3 8 1 0 5 1 3 1 5 5 6 1 8 5 1 7 9 4 6 2 2 5 0 6 5 6 3
 7 2 0 8 8 5 4 1 1 4 0 3 3 7 6 1 6 2 1 9 2 8 6 1 9 5 2 5 4 4 2 8 3 8 2 4 5
 0 3 1 7 7 5 7 9 7 1 9 2 1 4 2 9 2 0 4 9 1 4 8 1 8 4 5 9 8 8 3 7 6 0 0 3 0
 2 6 6 4 9 3 3 3 2 3 9 1 2 6 8 0 5 6 6 6 3 8

In [14]:
with open("test_labels.txt", 'w') as f:
    for label in test_targets.numpy():
        f.write(f"{label}\n")

In [15]:
# --- Check Accuracy on Test Subset ---
print("\n--- Test Subset Accuracy Check ---")
classifier.eval()

with torch.no_grad():
    # Move test data to device
    test_data_device = test_data.to(device)
    test_targets_device = test_targets.to(device)
    
    # Get model predictions on test subset
    outputs = classifier(test_data_device)
    _, predicted = torch.max(outputs, 1)
    
    # Calculate accuracy
    correct_subset = (predicted == test_targets_device).sum().item()
    total_subset = test_targets_device.size(0)
    accuracy_subset = 100 * correct_subset / total_subset
    
    print(f"Test Subset Accuracy: {accuracy_subset:.2f}% ({correct_subset}/{total_subset})")
    
    # Show detailed results for each sample
    print("\nDetailed Results:")
    print("Sample | True Label | Predicted | Correct?")
    print("-" * 40)
    
    for i in range(total_subset):
        true_label = test_targets[i].item()
        pred_label = predicted[i].item()
        is_correct = "✓" if true_label == pred_label else "✗"
        print(f"{i:6d} | {true_label:10d} | {pred_label:9d} | {is_correct}")
    
    # Show output scores for first few samples
    print(f"\nOutput scores for first 5 samples:")
    for i in range(min(5, total_subset)):
        scores = outputs[i].cpu().numpy()
        true_label = test_targets[i].item()
        pred_label = predicted[i].item()
        print(f"Sample {i} (True: {true_label}, Pred: {pred_label}):")
        print(f"  Scores: {scores}")
        print(f"  Max score: {scores.max():.3f} at index {scores.argmax()}")
        print()

print("------")
print("Analysis complete!")


--- Test Subset Accuracy Check ---


Test Subset Accuracy: 97.81% (626/640)

Detailed Results:
Sample | True Label | Predicted | Correct?
----------------------------------------
     0 |          7 |         7 | ✓
     1 |          2 |         2 | ✓
     2 |          1 |         1 | ✓
     3 |          0 |         0 | ✓
     4 |          4 |         4 | ✓
     5 |          1 |         1 | ✓
     6 |          4 |         4 | ✓
     7 |          9 |         9 | ✓
     8 |          5 |         5 | ✓
     9 |          9 |         9 | ✓
    10 |          0 |         0 | ✓
    11 |          6 |         6 | ✓
    12 |          9 |         9 | ✓
    13 |          0 |         0 | ✓
    14 |          1 |         1 | ✓
    15 |          5 |         5 | ✓
    16 |          9 |         9 | ✓
    17 |          7 |         7 | ✓
    18 |          3 |         3 | ✓
    19 |          4 |         4 | ✓
    20 |          9 |         9 | ✓
    21 |          6 |         6 | ✓
    22 |          6 |         6 | ✓
    23 |          5 |         